In [8]:
import os
import sys
from unittest import loader
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

TensorFlow version: 2.21.0
Num GPUs Available: 0


In [9]:
PROJECT_ROOT = r"D:\AI-Powered-Diabetic-Retinopathy-System"

CSV_PATH = os.path.join(
    PROJECT_ROOT,
    "ml-model",
    "dataset",
    "raw",
    "train.csv"
)

IMAGE_DIR = os.path.join(
    PROJECT_ROOT,
    "ml-model",
    "dataset",
    "raw",
    "train_images"
)

BASELINE_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "ml-model",
    "saved_models",
    "best_resnet50.keras"
)

FINETUNED_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "ml-model",
    "saved_models",
    "resnet50_finetuned.keras"
)

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Baseline model:", BASELINE_MODEL_PATH)

CSV: D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\dataset\raw\train.csv
Images: D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\dataset\raw\train_images
Baseline model: D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\best_resnet50.keras


In [33]:

preprocessing_dir = os.path.join(
    PROJECT_ROOT,
    "ml-model",
    "preprocessing"
)

if preprocessing_dir not in sys.path:
    sys.path.insert(0, preprocessing_dir)

from dataset_loader import (
    load_metadata,
    create_image_paths,
    validate_image_paths,
    split_dataset,
    create_tf_dataset
)

print("Dataset utilities imported successfully.")

Dataset utilities imported successfully.


In [34]:
df = load_metadata(CSV_PATH)

df = create_image_paths(df, IMAGE_DIR)

df = validate_image_paths(df)

print("Total valid images:", len(df))
print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())

Total valid images: 3662

Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [35]:
train_df, val_df, test_df = split_dataset(
    df,
    test_size=0.15,
    validation_size=0.15,
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2563
Validation: 549
Test: 550


In [36]:
BATCH_SIZE = 8

train_dataset = create_tf_dataset(
    train_df,
    batch_size=BATCH_SIZE,
    shuffle=True,
    training=True
)

val_dataset = create_tf_dataset(
    val_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    training=False
)

test_dataset = create_tf_dataset(
    test_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    training=False
)

print("Datasets created successfully.")

Datasets created successfully.


In [37]:
classes = np.sort(train_df["diagnosis"].unique())

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"]
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

print("Class weights:")
print(class_weights)

Class weights:
{0: 0.4058590657165479, 1: 1.9791505791505792, 2: 0.7333333333333333, 3: 3.797037037037037, 4: 2.476328502415459}


In [38]:
model = tf.keras.models.load_model(
    BASELINE_MODEL_PATH
)

print("Baseline ResNet50 loaded successfully.")
print("Model name:", model.name)

Baseline ResNet50 loaded successfully.
Model name: ResNet50_DR_Classifier


In [39]:
base_model = model.get_layer("resnet50")

print("Total ResNet50 layers:", len(base_model.layers))

print("\nLast 20 layers:")
for i, layer in enumerate(base_model.layers[-20:], start=len(base_model.layers)-20):
    print(i, layer.name, layer.__class__.__name__)

Total ResNet50 layers: 175

Last 20 layers:
155 conv5_block2_1_conv Conv2D
156 conv5_block2_1_bn BatchNormalization
157 conv5_block2_1_relu Activation
158 conv5_block2_2_conv Conv2D
159 conv5_block2_2_bn BatchNormalization
160 conv5_block2_2_relu Activation
161 conv5_block2_3_conv Conv2D
162 conv5_block2_3_bn BatchNormalization
163 conv5_block2_add Add
164 conv5_block2_out Activation
165 conv5_block3_1_conv Conv2D
166 conv5_block3_1_bn BatchNormalization
167 conv5_block3_1_relu Activation
168 conv5_block3_2_conv Conv2D
169 conv5_block3_2_bn BatchNormalization
170 conv5_block3_2_relu Activation
171 conv5_block3_3_conv Conv2D
172 conv5_block3_3_bn BatchNormalization
173 conv5_block3_add Add
174 conv5_block3_out Activation


In [40]:
base_model.trainable = False

print("All ResNet50 layers frozen.")

All ResNet50 layers frozen.


In [41]:
for layer in base_model.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

In [42]:
trainable_layers = [
    layer for layer in base_model.layers
    if layer.trainable
]

frozen_layers = [
    layer for layer in base_model.layers
    if not layer.trainable
]

print("Trainable backbone layers:", len(trainable_layers))
print("Frozen backbone layers:", len(frozen_layers))

print("\nTrainable layers:")
for layer in trainable_layers:
    print(layer.name)

Trainable backbone layers: 15
Frozen backbone layers: 160

Trainable layers:
input_layer
conv5_block2_1_conv
conv5_block2_1_relu
conv5_block2_2_conv
conv5_block2_2_relu
conv5_block2_3_conv
conv5_block2_add
conv5_block2_out
conv5_block3_1_conv
conv5_block3_1_relu
conv5_block3_2_conv
conv5_block3_2_relu
conv5_block3_3_conv
conv5_block3_add
conv5_block3_out


In [43]:
trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.trainable_weights
])

non_trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.non_trainable_weights
])

print("Trainable parameters:", trainable_params)
print("Non-trainable parameters:", non_trainable_params)

Trainable parameters: 8929285
Non-trainable parameters: 14668672


In [44]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model recompiled for fine-tuning.")

Model recompiled for fine-tuning.


In [45]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    FINETUNED_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

In [46]:
history_finetune = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        checkpoint,
        early_stopping
    ]
)

Epoch 1/5


d:\AI-Powered-Diabetic-Retinopathy-System\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step - accuracy: 0.7323 - loss: 0.8849
Epoch 1: val_loss improved from None to 0.76923, saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_finetuned.keras

Epoch 1: finished saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_finetuned.keras
321/321 ━━━━━━━━━━━━━━━━━━━━ 169s 511ms/step - accuracy: 0.7323 - loss: 0.8849 - val_accuracy: 0.6958 - val_loss: 0.7692
Epoch 2/5
321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.7433 - loss: 0.8427
Epoch 2: val_loss did not improve from 0.76923
321/321 ━━━━━━━━━━━━━━━━━━━━ 171s 532ms/step - accuracy: 0.7433 - loss: 0.8427 - val_accuracy: 0.6903 - val_loss: 0.8014
Epoch 3/5
321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.7569 - loss: 0.8064
Epoch 3: val_loss improved from 0.76923 to 0.69526, saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_finetuned.keras

Epoch 3: finished saving

In [47]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import tensorflow as tf

# Load the best fine-tuned checkpoint
finetuned_model = tf.keras.models.load_model(
    FINETUNED_MODEL_PATH
)

print("Fine-tuned model loaded successfully.")

Fine-tuned model loaded successfully.


In [48]:
# Generate predictions on the exact same test set
y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = finetuned_model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Test samples:", len(y_true))

Test samples: 550


In [49]:
class_names = [
    "No DR",
    "Mild DR",
    "Moderate DR",
    "Severe DR",
    "Proliferative DR"
]

print("Fine-tuned ResNet50 Classification Report")
print("---------------------------------------------")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

Fine-tuned ResNet50 Classification Report
---------------------------------------------
                  precision    recall  f1-score   support

           No DR     0.9565    0.9742    0.9653       271
         Mild DR     0.5000    0.6607    0.5692        56
     Moderate DR     0.7755    0.5067    0.6129       150
       Severe DR     0.3171    0.4483    0.3714        29
Proliferative DR     0.3770    0.5227    0.4381        44

        accuracy                         0.7509       550
       macro avg     0.5852    0.6225    0.5914       550
    weighted avg     0.7806    0.7509    0.7554       550



In [50]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix")
print("----------------")
print(cm)

Confusion Matrix
----------------
[[264   6   1   0   0]
 [  4  37  11   0   4]
 [  7  26  76  19  22]
 [  0   1   3  13  12]
 [  1   4   7   9  23]]
